# Notebook 00 -- Python & Tools Primer for AI Engineering

**Purpose:** This notebook covers the Python features, data formats, and command-line skills you will use throughout the entire curriculum. If you already know Python well, skim through and move to Notebook 01.

**Why this exists:** The notebooks ahead use decorators, async/await, type hints, SQL queries, JSON schemas, and Bash commands. Encountering these for the first time INSIDE a complex MLOps notebook is overwhelming. This primer lets you see each pattern in isolation first.

---

## What You Will Learn

| Skill | Where Used in Curriculum |
|-------|------------------------|
| Decorators (`@`) | Spark `@dp.table`, Ray `@ray.remote`, MCP `@server.tool()` |
| Type hints | Pydantic models (FastAPI), function signatures |
| `async` / `await` | LLM streaming, MCP servers, FastAPI endpoints |
| JSON parsing | MCP JSON-RPC, API requests/responses, config files |
| SQL basics | Flink SQL, database queries, MCP tools |
| Bash commands | Docker, kubectl, git, Linux navigation |
| Error handling | Production APIs, pipeline retry logic |


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors learn language syntax; seniors learn execution models. AI tools (FastAPI, Ray, PyTorch) heavily use decorators and async to manage concurrency and distributed state. Mastering these is non-negotiable for reading production code.

**Mental Model:** Think of decorators as 'middleware' for functions, and async/await as a restaurant kitchen where the chef (CPU) starts multiple delayed tasks (network requests) without standing around waiting for them to finish.

**Common Junior Pitfall:** Mixing synchronous (blocking) code inside async functions, which defeats the entire purpose and freezes the application under load.

---


In [ ]:
# Cell 1 -- Decorators: What does @ do?
#
# A decorator WRAPS a function to add behavior without changing its code.
# You will see @ray.remote, @dp.table, @app.post, @server.tool -- all decorators.

import time

# A simple decorator that measures execution time
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f'  {func.__name__} took {elapsed:.4f}s')
        return result
    return wrapper

@timer  # <-- This is the decorator. It wraps process_data with timing logic.
def process_data(n):
    """Simulate data processing."""
    total = sum(range(n))
    return total

# When you call process_data, the @timer wrapper runs automatically
result = process_data(1_000_000)
print(f'  Result: {result:,}')

# In the curriculum, decorators register functions with frameworks:
# @ray.remote      -> tells Ray this function can run on any worker
# @dp.table        -> tells Spark this function defines a dataset
# @app.post('/v1') -> tells FastAPI this handles POST requests to /v1
print('\nDecorator cheat sheet:')
print('  @ray.remote    = "Run this on any machine in the cluster"')
print('  @dp.table      = "This function defines a dataset"')
print('  @app.post(url) = "Handle HTTP POST requests at this URL"')
print('  @server.tool() = "Expose this function as an MCP tool"')


---
## 2 -- Type Hints & Pydantic Validation

### What are type hints?
Type hints tell Python (and other developers) what data types a function expects. Python does NOT enforce them at runtime -- but libraries like **Pydantic** DO.

```python
# Without type hints (what does this function expect?):
def process(data, threshold, output):
    pass

# With type hints (crystal clear!):
def process(data: list[float], threshold: float = 0.5, output: str = 'json') -> dict:
    pass
```

### Why Pydantic Matters in AI Engineering

In production AI APIs, BAD INPUT is the #1 cause of crashes. Pydantic validates every field automatically:
- User sends `temperature: "hot"` instead of `temperature: 0.7` -> Pydantic rejects it BEFORE your model runs
- User sends `age: -5` -> Pydantic catches it if you set `ge=0`


In [ ]:
# Cell 2 -- Pydantic validation (used in FastAPI, MCP, everywhere)
from pydantic import BaseModel, Field
from typing import List, Optional

class InferenceRequest(BaseModel):
    """Every field is validated automatically."""
    prompt: str = Field(..., min_length=1, max_length=4096, description='The user prompt')
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=256, ge=1, le=4096)
    model: str = Field(default='gpt-4o')
    tags: Optional[List[str]] = None

# Valid request
req = InferenceRequest(prompt='Explain quantum computing')
print(f'Valid request: {req.model_dump()}')

# Invalid request -- Pydantic catches the errors automatically
try:
    bad = InferenceRequest(prompt='', temperature=5.0, max_tokens=-1)
except Exception as e:
    print(f'\nPydantic caught errors:\n{e}')
    print('\nWhy this matters: Without Pydantic, this bad input reaches your GPU,')
    print('causes a cryptic CUDA error, and you spend 3 hours debugging.')


---
## 3 -- Async / Await: Doing Multiple Things at Once

### The Problem Async Solves

Imagine you're a chef. **Synchronous cooking:**
```
1. Put bread in toaster, WAIT 3 min (staring at toaster)
2. Boil water, WAIT 5 min (staring at pot)
3. Fry egg, WAIT 2 min
Total: 10 minutes (doing one thing at a time)
```

**Async cooking:**
```
1. Put bread in toaster (don't wait)
2. Start boiling water (don't wait)
3. Fry egg while toaster and pot work
Total: 5 minutes (everything runs in parallel)
```

In AI backends, your server calls LLM APIs that take 2-30 seconds. Without async, your server can only handle ONE user at a time. With async, it handles THOUSANDS.

### The Two Keywords
| Keyword | Meaning | Analogy |
|---------|---------|--------|
| `async def` | This function CAN be paused | A chef who multitasks |
| `await` | Pause HERE until result arrives | "I'll come back when the toast is done" |


In [ ]:
# Cell 3 -- Async/await demo
import asyncio

# SYNCHRONOUS: blocks on each call
def sync_chef():
    import time
    start = time.time()
    time.sleep(0.3); print('  Toast done')
    time.sleep(0.5); print('  Water boiled')
    time.sleep(0.2); print('  Egg fried')
    print(f'  Sync total: {time.time()-start:.1f}s')

# ASYNCHRONOUS: runs in parallel
async def async_chef():
    import time
    start = time.time()
    await asyncio.gather(
        asyncio.sleep(0.3),  # Toast
        asyncio.sleep(0.5),  # Boil water
        asyncio.sleep(0.2),  # Fry egg
    )
    print(f'  Async total: {time.time()-start:.1f}s (all ran in parallel!)')

print('Synchronous (one at a time):')
sync_chef()
print('\nAsynchronous (parallel):')
asyncio.get_event_loop().run_until_complete(async_chef())

print('\nIn AI backends:')
print('  sync:  1 user calls LLM -> server WAITS 5s -> can\'t serve anyone else')
print('  async: 1000 users call LLM -> server awaits ALL -> serves everyone')


---
## 4 -- JSON: The Universal Data Format

JSON is used EVERYWHERE in AI engineering:
- API requests and responses (OpenAI, Anthropic, every LLM provider)
- MCP messages (JSON-RPC 2.0)
- Configuration files
- Logging and monitoring

### The JSON-RPC Format (Used in MCP)
```json
{"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "query_db", "arguments": {"sql": "SELECT *"}}, "id": 1}
```
This is just a dictionary with specific required keys. Once you understand JSON parsing, MCP becomes much less mysterious.


In [ ]:
# Cell 4 -- JSON operations you'll use constantly
import json

# Parse a JSON string (API response)
api_response = '{"id": "chatcmpl-123", "choices": [{"message": {"content": "Hello!"}, "finish_reason": "stop"}], "usage": {"prompt_tokens": 10, "completion_tokens": 5}}'
data = json.loads(api_response)

# Access nested fields (VERY common in LLM APIs)
content = data['choices'][0]['message']['content']
tokens_used = data['usage']['prompt_tokens'] + data['usage']['completion_tokens']
print(f'Content: {content}')
print(f'Tokens used: {tokens_used}')

# Build a JSON-RPC request (MCP format)
mcp_request = {
    'jsonrpc': '2.0',
    'method': 'tools/call',
    'params': {
        'name': 'query_database',
        'arguments': {'sql': 'SELECT COUNT(*) FROM orders WHERE status = \'pending\''}
    },
    'id': 42
}
print(f'\nMCP Request:\n{json.dumps(mcp_request, indent=2)}')

# Pretty-print vs compact (for logging vs API calls)
print(f'\nCompact: {json.dumps(data, separators=(".",":"))[:80]}...')
print(f'Pretty: {json.dumps(data, indent=2)[:120]}...')


---
## 5 -- SQL Crash Course

SQL appears in multiple notebooks:
- **Flink SQL** (NB05) for streaming data processing
- **MCP tools** (NB12) for database queries
- **Feature stores** (NB06) for feature extraction

### The 5 SQL Statements You Need
```sql
SELECT columns FROM table WHERE condition;           -- Read data
SELECT col, COUNT(*) FROM table GROUP BY col;         -- Aggregate
SELECT a.*, b.name FROM orders a JOIN users b ON a.user_id = b.id;  -- Combine tables
SELECT *, AVG(price) OVER (PARTITION BY category) FROM products;     -- Window function
INSERT INTO table (col1, col2) VALUES (val1, val2);   -- Write data
```

### Flink SQL Is Just SQL on Streams
Instead of querying a static table, you query data FLOWING through a pipe:
```sql
-- Regular SQL: counts all orders ever
SELECT user_id, COUNT(*) FROM orders GROUP BY user_id;

-- Flink SQL: counts orders in 5-minute sliding windows (LIVE)
SELECT user_id, COUNT(*) FROM orders GROUP BY user_id, TUMBLE(event_time, INTERVAL '5' MINUTE);
```


In [ ]:
# Cell 5 -- SQL with Python (sqlite3 -- built in, no install needed)
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE orders (id INT, user_id INT, product TEXT, amount REAL, status TEXT)')
orders = [(1,'u1','laptop',999.99,'completed'), (2,'u1','mouse',29.99,'completed'),
          (3,'u2','keyboard',79.99,'pending'), (4,'u3','monitor',349.99,'completed'),
          (5,'u2','laptop',1099.99,'completed'), (6,'u3','cable',12.99,'pending')]
conn.executemany('INSERT INTO orders VALUES (?,?,?,?,?)', orders)

# SELECT + WHERE (filter)
print('1. SELECT with WHERE:')
print(pd.read_sql('SELECT * FROM orders WHERE amount > 100', conn).to_string(index=False))

# GROUP BY + aggregate
print('\n2. GROUP BY (aggregate per user):')
print(pd.read_sql('SELECT user_id, COUNT(*) as order_count, SUM(amount) as total_spent FROM orders GROUP BY user_id', conn).to_string(index=False))

# Window function (running total)
print('\n3. Window function (running total per user):')
print(pd.read_sql('SELECT user_id, product, amount, SUM(amount) OVER (PARTITION BY user_id ORDER BY id) as running_total FROM orders', conn).to_string(index=False))
conn.close()


---
## 6 -- Bash / Terminal Commands

You will use the terminal for Docker, Kubernetes, Git, and server management. Here are the essential commands:

```bash
# Navigation
ls -la          # List files with details
cd /path/to/dir # Change directory
pwd             # Print current directory

# Pipelines (chain commands)
cat file.log | grep 'ERROR' | wc -l    # Count error lines in a log
#  cat = read file | grep = filter lines | wc -l = count lines

# Docker (container management)
docker build -t my-app .          # Build image from Dockerfile
docker run -p 8000:8000 my-app    # Run container, expose port 8000
docker ps                          # List running containers

# Kubernetes (orchestration)
kubectl get pods                   # List running pods
kubectl apply -f deploy.yaml       # Apply configuration
kubectl logs pod-name              # View pod logs

# Git (version control)
git add . && git commit -m 'msg'   # Stage and commit changes
git push origin main               # Push to remote
```


In [ ]:
# Cell 6 -- Error handling patterns (production APIs MUST handle errors)
import random

def call_llm_api(prompt, fail_rate=0.3):
    """Simulate an LLM API that sometimes fails (just like real APIs)."""
    if random.random() < fail_rate:
        raise ConnectionError('API timeout after 30s')
    return f'Response to: {prompt[:30]}'

# BAD: No error handling (crashes your entire server)
# result = call_llm_api('Hello')  # If this fails, server returns 500

# GOOD: Try/except with retry logic
def call_with_retry(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            result = call_llm_api(prompt)
            print(f'  Attempt {attempt+1}: Success -> {result}')
            return result
        except ConnectionError as e:
            print(f'  Attempt {attempt+1}: Failed ({e})')
            if attempt == max_retries - 1:
                print(f'  All {max_retries} attempts failed. Returning fallback.')
                return 'Sorry, service temporarily unavailable.'

random.seed(42)
print('API call with retry pattern:')
call_with_retry('Explain machine learning')
print('\nThis retry pattern appears in: circuit breakers (NB17), CI/CD (NB09), deployment (NB11)')


---
## Summary

| Python Feature | Where It Appears | Why It Matters |
|---------------|-----------------|----------------|
| Decorators `@` | Ray, Spark, FastAPI, MCP | Registers functions with frameworks |
| Type hints | Pydantic, all API schemas | Catches bad inputs before GPU execution |
| `async/await` | FastAPI, MCP, streaming | Serves thousands of users simultaneously |
| JSON | Every API, MCP, logging | Universal data exchange format |
| SQL | Flink, MCP tools, feature stores | Querying data at scale |
| Bash | Docker, K8s, Git, server management | Operating production infrastructure |
| Try/except | APIs, pipelines, agents | Prevents one failure from crashing everything |

**Next -->** `01_infrastructure_orchestration.ipynb` -- Linux CLI, Git, Docker, Kubernetes